# പാഠം 05 - ഏജൻസിക് RAG


## സെറ്റ് അപ്

ഈ നോട്ട്‌ബുക്ക് Microsoft Agent Framework ഉപയോഗിച്ച് Agentic RAG (Retrieval-Augmented Generation) പാറ്റേൺ കാണിക്കുന്നു.

**ആവശ്യമായവ:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — നിങ്ങളുടെ Azure AI Search സർവീസ് എന്റ്പോയിന്റ്
- `AZURE_SEARCH_API_KEY` — നിങ്ങളുടെ Azure AI Search API കീ
- എൻവയോൺമെൻറ് വ്യത്യാസങ്ങളിലൂടെ ക്രമീകരിച്ചത് Azure OpenAI ഡിപ്ലോയ്മെന്റ്
- Azure CLI അഥെൻറ്റിക്കേറ്റഡ് (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ഏജന്റിക് RAG എന്താണ്?

സംപ്രേഷണ RAG ഒരു നിശ്ചിത പൈപ്പ്‌ലൈൻ പിന്തുടരുന്നു: പ്രമാണങ്ങൾ കണ്ടെത്തുക, പിന്നെ ഒരു പ്രതികരണം സൃഷ്ടിക്കുക. **ഏജന്റിക് RAG** അതിനോട് കൂടുതൽ മുന്നേറുന്നു, ഏജന്റിന് **എപ്പോഴാണ്** എന്നതും **എങ്ങനെ** എന്നതും വിവരങ്ങൾ കണ്ടെത്തേണ്ടതെന്ന് സ്വതന്ത്രമായി തീരുമാനിക്കാൻ കഴിവ് നൽകുന്നു.

ഏജന്റിക് RAG ഉപയോഗിച്ച്, ഏജന്റ് കഴിയും:
- ഒരു ചോദ്യത്തിന് മറുപടി നൽകുന്നതിന് മുന്‍പ് കണ്ടെത്തല്‍ ആവശ്യമായാണോ എന്നു **തീരുമാനിക്കുക**
- ഏതു ഡാറ്റ സ്രോതസ്സോ ഉപകരണമോ പരിശോധിക്കണമെന്ന് **തെരഞ്ഞെടുത്തുക**
- കണ്ടെത്തിയ ഫലങ്ങൾ **അവലോകനം ചെയ്ത്** ആദ്യ ശ്രമം പ്രയോജനപ്പെടുത്തുന്നതിന് മുമ്പ് തുടർന്നുള്ള കണ്ടെത്തലുകൾ നടത്തുക
- പല കണ്ടെത്തൽ ഘട്ടങ്ങളിൽ നിന്നുള്ള വിവരങ്ങളെ ഒന്നിച്ച് ചേർത്ത് സുപ്രസ്തുതമായ ഒരു ഉത്തരം **സൃഷ്ടിക്കുക**

ഇതി నుంచి ഏജന്റ് സ്ഥിരം കണ്ടെത്തി-പിന്നീട് സൃഷ്ടിക്കുക പൈപ്പ്‌ലൈനുമായി താരതമ്യപ്പെടുത്തുമ്പോൾ കൂടുതൽ ചമതികളോടും കൃത്യതയോടും ആയിരിക്കും.


## Creating a Search Tool

In Agentic RAG, external data sources are wrapped as **tools** that the agent can invoke on demand. This lets the agent treat retrieval as just another action it can take, rather than a mandatory step.

Below we define a travel knowledge base and expose it as a tool the agent can call to look up destination information.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ഏജന്റ് നിർമ്മാണം

ഇപ്പോൾ ഞങ്ങൾ ഒരു ഏജന്റ് സൃഷ്ടിക്കുന്നു, ഇത് **പുമ്പോൾ ഉത്തരം നൽകുന്നതിന് മുമ്പ് എപ്പോഴും വിവരങ്ങൾ തിരയാൻ** നിർദ്ദേശിക്കപ്പെട്ടതാണ്. ഏജന്റ് തന്റെ ഉറവിടം പരിശീലന ഡാറ്റയിൽ ആശ്രയിക്കാൻ പകരം, തന്റെ പ്രതികരങ്ങൾ അറിവ് അടിസ്ഥാനത്തിൽ ഉറപ്പാക്കാൻ `search_travel_knowledge` ഉപകരണത്തെ ഉപയോഗിക്കുന്നു.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## പുരവർത്തിത തിരയൽ — മേക്കർ-ചെക്കർ പാറ്റേൺ

Agentic RAG-ന്റെ പ്രധാന കരുത്ത് **പുരവർത്തിത തിരയലാണ്**. ഏജന്റ് തുടക്കത്തിലുള്ള കണ്ടെത്തലുകൾ സ്ഥിരീകരിക്കൽ, ശരിയാക്കൽ, അല്ലെങ്കിൽ വിപുലീകരിക്കൽ നടത്തുന്നതിന് പല തവണ തിരച്ചിൽ നടത്താൻ കഴിയും — ഇത് ഒരു "മേക്കർ-ചെക്കർ" വർക്ക്‌ഫ്ലോ പോലെ ആണ്:

1. **മേക്കർ ഘട്ടം**: ഏജന്റ് തുടക്കക്കാരിയായ വിവരങ്ങൾ തിരഞ്ഞെടുത്തു ഒരു മറുപടി തയ്യാറാക്കുന്നു.
2. **ചെക്കർ ഘട്ടം**: ഏജന്റ് കൂടുതൽ വിവരങ്ങൾ സ്ഥിരീകരിക്കാനും പിഴവുകൾ പൂരിപ്പിക്കാനുമുള്ള അധിക തിരച്ചിലുകൾ നടത്തുന്നു.

താഴെ, ഏജന്റിന് കയറ്റം ഒരു ചോദ്യമാണ് ചോദിച്ചത്, ഇത് പല സഞ്ചാരണങ്ങളിൽ വ്യത്യസ്ത സ്ഥലങ്ങൾ താരതമ്യം ചെയ്യുന്നതിന് ആവശ്യമാണ്, അതിനാൽ ഏജന്റ് പല തവണ തിരയാൻ പ്രേരിപ്പിക്കുന്നു.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

ഈ പാഠത്തിൽ നിങ്ങൾ Microsoft Agent Framework ഉപയോഗിച്ച് **Agentic RAG** സിസ്റ്റം എങ്ങനെ നിർമ്മിക്കാമെന്ന് പഠിച്ചു:

- **Agentic RAG** ഏജന്റുകൾ സ്വയം എപ്പോൾ വിവരങ്ങൾ തിരയണമെന്ന് തീരുമാനിക്കാൻ അനുവദിക്കുന്നു, അതിലൂടെ തിരയൽ സ്ഥിരമായതിനെക്കാൾ പ്രാപ്തമായിരിക്കും.
- **ഉപകരണങ്ങൾ ഡാറ്റാ സ്രോതസുകളായി**: എജന്റ് വിളിക്കാവുന്ന ഉപകരണങ്ങളായി പുറം ജ്ഞാനഭണ്ഡാരങ്ങൾ (Azure AI Search പോലുള്ളവ) പൊതിഞ്ഞിരിക്കുന്നു.
- **ത്രൈമാസിക തിരയൽ**: മേക്കർ-ചെക്കർ മാതൃക എജന്റിന് ഒന്നിലധികം തിരയൽ പര്യവേഷണങ്ങൾ (തിരയൽ, സ്ഥിരീകരണം, മെച്ചപ്പെടുത്തൽ) നടത്താൻ സാധിക്കുന്നതാക്കുന്നു, പിന്നീട് അന്തിമ ഉത്തരം നൽകുന്നു.

ഉൽപ്പാദന സാഹചര്യത്തിൽ, വലിയ തോതിലുള്ള യാത്രാ ഡോക്യുമെന്റ് തിരയൽ കൈകാര്യം ചെയ്യാൻ ഇൻ-മെമ്മറി `TRAVEL_KNOWLEDGE_BASE` യെ യഥാർത്ഥ Azure AI Search ഇൻഡക്സിൽ മാറ്റും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അസ്വീകരണം**:  
ഈ ഡോക്യുമെന്റ് [കോ-ഓപ് ട്രാൻസ്ഡേറ്റർ](https://github.com/Azure/co-op-translator) എന്ന AI വിവർത്തന സേവനം ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയിട്ടുണ്ട്. ഞങ്ങൾ ശരിയായ വിവർത്തനത്തിന് ശ്രമിച്ചിട്ടും, യാന്ത്രിക വിവർത്തനങ്ങളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റുകൾ ഉണ്ടായേക്കാവുന്നുവെന്ന് ദയവായി ശ്രദ്ധിക്കുക. പ്രാഥമിക ഭാഷയിലുള്ള(original) ഡോക്യുമെന്റാണ് ഔദ്യോഗിക ഉറവിടമെന്നാണ് പരിഗണിക്കേണ്ടത്. നിർണായക വിവരങ്ങൾക്ക് വിദഗ്ധ മനുഷ്യ വിവർത്തനം ശുപാർശ ചെയ്യപ്പെടുന്നു. ഈ വിവർത്തനം ഉപയോഗിച്ച് ഉണ്ടാകാനിടയുള്ള ഏതൊരു തെറ്റിദ്ധാരണക്കും അല്ലെങ്കിൽ തെറ്റായി വ്യാഖ്യാനങ്ങൾക്കുമുള്ള ഉത്തരവാദിത്തം ഞങ്ങൾ ഏറ്റെടുക്കുന്നില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
